# Supplementary experiments

Six small, self-contained follow-up checks that used to be separate `.py`
files (`scan_sl_routes.py`, `step4k_rf_feature_importance.py`,
`step4l_multiseed_mlp.py`, `step4m_alt_target_activity.py`,
`step4n_gatv2_edge_attrs.py`, `step4o_tabular_ensemble.py`). Same code, same
logic -- consolidated here so there's one clean file instead of six loose
scripts to track.

**None of these are the headline result.** `step4_model.py --with-sc` is
that (see `RUNBOOK.md`). These are supplementary checks; their results are
already committed as CSVs (`results_rf_feature_importance.csv`,
`results_cv_multiseed_mlp.csv`, etc.) -- only re-run a section if you want
to redo that specific check from scratch.

**Sections are independent -- run any one, skip the rest, any order.**
Each explicitly reloads `step4_model` with the flags it needs, so running
one section never leaks its configuration into another (Python caches
imports; without the reload, a later section could silently inherit an
earlier one's settings).

Set `QUICK_MODE = True` below to smoke-test every section on 5 boroughs
instead of 33 (minutes instead of hours). Sections 3 and 4 (alt-target,
edge-attrs) need the raw `data/` folder -- see `RUNBOOK.md` Sec.4.0.

In [ ]:
QUICK_MODE = False  # True -> 5 boroughs per section instead of 33

import sys, time, glob, itertools, importlib, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor
from scipy.stats import wilcoxon

DATA_FILE   = "stops_features_osm.csv"
BOROUGH_COL = "lad_name"
TARGET_COL  = "total_boardings"

def log(msg):
    print(msg, flush=True)

def wmape(y_true, y_pred):
    return float(np.sum(np.abs(y_true - y_pred)) / (np.sum(np.abs(y_true)) + 1e-8))

def score(y_true, y_pred, name):
    y_pred = np.clip(y_pred, 0, None)
    return {"model": name, "WMAPE": round(wmape(y_true, y_pred), 4),
            "RMSE": round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 3),
            "MAE": round(float(mean_absolute_error(y_true, y_pred)), 3)}

def stratified_val_split(tr_idx, y_tr, seed, fi, val_frac):
    # same stratified-5-quantile split used everywhere in this pipeline
    rng = np.random.RandomState(seed + fi)
    n_val = max(5, int(len(tr_idx) * val_frac))
    quantile_labels = pd.qcut(np.log1p(y_tr), q=5, labels=False, duplicates="drop")
    vp_list, n_per_q = [], max(1, n_val // 5)
    for q in range(5):
        q_idx = np.where(quantile_labels == q)[0]
        if len(q_idx) > 0:
            vp_list.extend(rng.choice(q_idx, min(n_per_q, len(q_idx)), replace=False).tolist())
    vp = np.array(vp_list)
    tp = np.setdiff1d(np.arange(len(tr_idx)), vp)
    return tp, vp

def fresh_step4_model(*flags):
    """Import (or reload) step4_model with exactly the given CLI flags set,
    regardless of what any earlier section in this notebook already imported
    -- avoids the module-caching trap described above."""
    sys.argv = [sys.argv[0]] + list(flags)
    import step4_model as sm
    importlib.reload(sm)
    return sm


## 0. Superloop (SL) route scan

Checks whether SL-prefixed (Superloop) route codes appear in the raw BUSTO
extract -- the dissertation notes SL1/SL10 are excluded from BUSTO v1.0.
Needs the raw `data/` folder.

In [ ]:
import os

def scan_sl_routes():
    data_dir = "data"
    if not os.path.isdir(data_dir):
        log("DATA_DIR_NOT_FOUND -- skipping (needs the raw data/ folder)")
        return

    files = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.lower().endswith(".csv")]
    if not files:
        log("NO_CSV_FILES_IN_DATA_FOLDER")
        return

    results = {}
    for f in files:
        log(f"\n--- Scanning file: {os.path.basename(f)}")
        try:
            df0 = pd.read_csv(f, nrows=0)
            cols = df0.columns.tolist()
            route_col = next((c for c in cols if c.strip().lower() == "route"), None)
            if route_col is None:
                route_col = next((c for c in cols if "route" in c.strip().lower()), None)
            if route_col is None:
                log("  NO_ROUTE_COLUMN_FOUND")
                results[os.path.basename(f)] = {"route_col": None, "matches": 0, "unique_codes": []}
                continue
            log(f"  Found route column: {route_col}")

            chunk_iter = pd.read_csv(f, usecols=[route_col], chunksize=200_000,
                                      dtype={route_col: str}, low_memory=True)
            total_matches, codes = 0, set()
            for chunk in chunk_iter:
                s = chunk[route_col].astype(str).str.upper().str.strip()
                mask = s.str.startswith("SL")
                if mask.any():
                    total_matches += int(mask.sum())
                    codes.update(s[mask].unique().tolist())
            log(f"  Matches: {total_matches}")
            if codes:
                log(f"  Unique matching route codes: {sorted(codes)}")
            results[os.path.basename(f)] = {"route_col": route_col, "matches": total_matches,
                                             "unique_codes": sorted(codes)}
        except Exception as e:
            log(f"  ERROR reading file: {e}")
            results[os.path.basename(f)] = {"route_col": None, "error": str(e)}

    log("\n=== SUMMARY ===")
    for fn, info in results.items():
        log(f"{fn} {info}")
    return results

scan_sl_routes()


## 1. Random forest feature importance (AI23-only)

Retrains RF once on the AI23-only feature set (8 AI23 columns + lat/lon),
same hyperparameters as the headline pipeline, and saves
`.feature_importances_` per fold -- backs up the claim that `main_bua_30min`
barely matters. Fast (RF only, no neural nets).

In [ ]:
def run_step4k():
    sm = fresh_step4_model()  # no flags needed -- FEAT_COLS/COORD_COLS/RF_TREES/SEED aren't flag-dependent
    active_feat = sm.FEAT_COLS + sm.COORD_COLS   # 8 AI23 + lat + lon = 10 features

    df = pd.read_csv(DATA_FILE)
    y_orig = df[TARGET_COL].values.astype(float)
    X_raw = np.zeros((len(df), len(active_feat)), dtype=np.float32)
    for i, c in enumerate(sm.FEAT_COLS):
        X_raw[:, i] = np.log1p(df[c].values)
    X_raw[:, len(sm.FEAT_COLS)] = df["lat"].values
    X_raw[:, len(sm.FEAT_COLS) + 1] = df["lon"].values
    log(f"Features ({len(active_feat)}): {active_feat}")

    boroughs = sorted(df[BOROUGH_COL].unique())
    if QUICK_MODE:
        boroughs = boroughs[:5]
    log(f"Running {len(boroughs)}-fold RF feature-importance pass...\n")

    rows = []
    t_start = time.time()
    for fi, b in enumerate(boroughs):
        t0 = time.time()
        test_mask = (df[BOROUGH_COL] == b).values
        tr_idx = np.where(~test_mask)[0]
        y_tr = y_orig[tr_idx]

        scaler = StandardScaler()
        X_sc_tr = scaler.fit_transform(X_raw[tr_idx])
        rf = RandomForestRegressor(n_estimators=sm.RF_TREES, max_features="sqrt",
                                    min_samples_leaf=5, n_jobs=-1, random_state=sm.SEED)
        rf.fit(X_sc_tr, np.log1p(y_tr))
        log(f"  [{fi+1:2d}/{len(boroughs)}] {b:<30s}  n_train={len(tr_idx):5d}  ({time.time()-t0:.1f}s)")
        rows.append({"borough": b, **dict(zip(active_feat, rf.feature_importances_))})

    imp_df = pd.DataFrame(rows).set_index("borough")
    imp_df.to_csv("results_rf_feature_importance_perfold.csv")

    summary = imp_df.agg(["mean", "std"]).T.sort_values("mean", ascending=False)
    summary.index.name = "feature"
    summary.to_csv("results_rf_feature_importance.csv")

    log(f"\n{'='*70}\nRF FEATURE IMPORTANCE (AI23-only, mean over {len(boroughs)} folds)\n{'='*70}")
    for feat, row in summary.iterrows():
        marker = "  <-- main_bua_30min" if feat == "main_bua_30min" else ""
        log(f"  {feat:<24s} mean={row['mean']:.4f}  std={row['std']:.4f}{marker}")
    log(f"\nTotal time: {(time.time()-t_start)/60:.1f} min")
    log("Saved -> results_rf_feature_importance.csv | results_rf_feature_importance_perfold.csv")
    return summary

run_step4k()


## 2. Multi-seed standalone MLP (noise floor)

Retrains the standalone MLP (no GATv2 -- MLP has no graph, trains fast) on
the headline AI23+OSM+SC feature set across 5 seeds (42, 142, 242, 342,
442), same 33 leave-borough-out folds. Reports mean +/- sd WMAPE across
seeds -- this is what establishes the "0.6301 +/- 0.0004" stability claim
for the headline MLP result.

In [ ]:
def run_step4l():
    sm = fresh_step4_model("--with-sc")   # force headline AI23+OSM+SC feature set
    SEEDS = [42, 142, 242, 342, 442]

    df = pd.read_csv(DATA_FILE)
    y_orig = df[TARGET_COL].values.astype(float)
    X_raw = sm.prep_features(df)
    log(f"Features: {X_raw.shape[1]} (AI23+OSM+SC headline set)")

    boroughs = sorted(df[BOROUGH_COL].unique())
    if QUICK_MODE:
        boroughs = boroughs[:5]
    log(f"Running {len(boroughs)} folds x {len(SEEDS)} seeds = {len(boroughs)*len(SEEDS)} MLP fits\n")

    all_rows = []
    t_start = time.time()
    for seed in SEEDS:
        log(f"--- seed={seed} ---")
        sm.SEED = seed
        torch.manual_seed(seed); np.random.seed(seed)
        for fi, b in enumerate(boroughs):
            t0 = time.time()
            test_mask = (df[BOROUGH_COL] == b).values
            tr_idx, te_idx = np.where(~test_mask)[0], np.where(test_mask)[0]
            y_tr, y_te = y_orig[tr_idx], y_orig[te_idx]

            scaler = sm.StandardScaler()
            X_sc_tr = scaler.fit_transform(X_raw[tr_idx])
            X_sc_te = scaler.transform(X_raw[te_idx])

            tp, vp = stratified_val_split(tr_idx, y_tr, seed, fi, sm.VAL_FRAC)
            tp_t, vp_t = torch.tensor(tp, dtype=torch.long), torch.tensor(vp, dtype=torch.long)
            x_tr = torch.tensor(X_sc_tr, dtype=torch.float)
            x_te = torch.tensor(X_sc_te, dtype=torch.float)
            y_tr_t = torch.tensor(np.log1p(y_tr), dtype=torch.float)

            mlp = sm.train_nn(sm.MLPModel(X_sc_tr.shape[1]), x_tr, y_tr_t, None, tp_t, vp_t)
            mlp.eval()
            with torch.no_grad():
                mlp_pred = np.clip(np.expm1(mlp(x_te).numpy()), 0, None)

            fold_wmape = wmape(y_te, mlp_pred)
            log(f"    [{fi+1:2d}/{len(boroughs)}] {b:<30s} n={len(te_idx):4d}  "
                f"WMAPE={fold_wmape:.4f}  ({time.time()-t0:.0f}s)")
            all_rows.append({"seed": seed, "borough": b, "n_test": len(te_idx),
                              "WMAPE": round(fold_wmape, 4)})

    result = pd.DataFrame(all_rows)
    suffix = "_quick" if QUICK_MODE else ""
    result.to_csv(f"results_cv_multiseed_mlp{suffix}.csv", index=False)

    per_seed = result.groupby("seed")["WMAPE"].mean()
    log(f"\n{'='*60}\nPer-seed mean WMAPE:")
    for s, w in per_seed.items():
        log(f"  seed={s:<5d} WMAPE={w:.4f}")
    log(f"{'='*60}\nAcross {len(SEEDS)} seeds: mean={per_seed.mean():.4f}  sd={per_seed.std():.4f}")
    log("(Frozen headline MLP, seed=42 only, for reference: 0.6311)")

    summary = pd.DataFrame({"seed": per_seed.index, "WMAPE_mean": per_seed.values})
    summary.to_csv(f"results_summary_multiseed_mlp{suffix}.csv", index=False)
    log(f"\nTotal time: {(time.time()-t_start)/60:.1f} min")
    return summary

run_step4l()


## 3. Alternative target: boardings + alightings

Retrains the full 7-model headline pipeline (AI23+OSM+SC, 33-fold LBO)
with target = `total_boardings + total_alightings` instead of boardings
alone, to check whether the headline conclusions are sensitive to that
choice. **Needs the raw `data/` folder** (reads `Alightings` from the raw
BUSTO CSVs -- that column isn't in any committed feature file).

In [ ]:
def build_total_activity(df):
    files = glob.glob("data/*Weekday*QUARTER HOUR*.csv")
    if not files:
        raise FileNotFoundError("Raw data/ folder not found -- this section needs it (see RUNBOOK.md Sec.4.0).")
    frames = [pd.read_csv(f, dtype={"STOPCODE": "string"}, usecols=["STOPCODE", "Alightings"])
              for f in files]
    raw = pd.concat(frames, ignore_index=True)
    raw["STOPCODE"] = raw["STOPCODE"].str.strip().str.upper()
    alight = raw.groupby("STOPCODE").agg(total_alightings=("Alightings", "sum")).reset_index()

    df = df.copy()
    df["STOPCODE_key"] = df["STOPCODE"].astype(str).str.strip().str.upper()
    df = df.merge(alight, left_on="STOPCODE_key", right_on="STOPCODE", how="left", suffixes=("", "_r"))
    df["total_alightings"] = df["total_alightings"].fillna(0.0)
    df["total_activity"] = df["total_boardings"] + df["total_alightings"]
    return df


def run_step4m():
    sm = fresh_step4_model("--with-sc")
    from torch_geometric.utils import subgraph as pyg_subgraph

    df = pd.read_csv(DATA_FILE)
    df = build_total_activity(df)
    log(f"total_activity built: mean={df['total_activity'].mean():.1f}  "
        f"(vs total_boardings mean={df['total_boardings'].mean():.1f}, "
        f"total_alightings mean={df['total_alightings'].mean():.1f})")

    y_orig = df["total_activity"].values.astype(float)
    X_raw = sm.prep_features(df)

    log(f"Building multigraph ({len(df):,} nodes)...")
    knn_ei = sm.build_knn_edge_index(df["lat"].values, df["lon"].values)
    route_ei = sm.build_route_edges(df)
    full_ei = torch.unique(torch.cat([knn_ei, route_ei], dim=1), dim=1)

    boroughs = sorted(df[BOROUGH_COL].unique())
    if QUICK_MODE:
        boroughs = boroughs[:5]
    log(f"Running {len(boroughs)}-fold leave-borough-out CV...\n")

    rows = []
    t_start = time.time()
    for fi, b in enumerate(boroughs):
        t0 = time.time()
        test_mask = (df[BOROUGH_COL] == b).values
        tr_idx, te_idx = np.where(~test_mask)[0], np.where(test_mask)[0]
        y_tr, y_te = y_orig[tr_idx], y_orig[te_idx]

        scaler = StandardScaler()
        X_sc_tr = scaler.fit_transform(X_raw[tr_idx])
        X_sc_te = scaler.transform(X_raw[te_idx])

        ha = score(y_te, np.full(len(te_idx), y_tr.mean()), "HistAvg")
        idw_pred = np.expm1(sm.idw_predict(df["lat"].values[tr_idx], df["lon"].values[tr_idx],
                                            np.log1p(y_tr),
                                            df["lat"].values[te_idx], df["lon"].values[te_idx]))
        idw_s = score(y_te, idw_pred, "IDW")

        mlr = Ridge(alpha=1.0, random_state=sm.SEED)
        mlr.fit(X_sc_tr, np.log1p(y_tr))
        mlr_s = score(y_te, np.expm1(mlr.predict(X_sc_te)), "MLR")

        rf = RandomForestRegressor(n_estimators=sm.RF_TREES, max_features="sqrt",
                                    min_samples_leaf=5, n_jobs=-1, random_state=sm.SEED)
        rf.fit(X_sc_tr, np.log1p(y_tr))
        rf_s = score(y_te, np.expm1(rf.predict(X_sc_te)), "RF")

        xgbr = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05,
                             random_state=sm.SEED, n_jobs=-1, verbosity=0)
        xgbr.fit(X_sc_tr, np.log1p(y_tr))
        xgb_s = score(y_te, np.expm1(xgbr.predict(X_sc_te)), "XGBoost")

        tp, vp = stratified_val_split(tr_idx, y_tr, sm.SEED, fi, sm.VAL_FRAC)
        tp_t, vp_t = torch.tensor(tp, dtype=torch.long), torch.tensor(vp, dtype=torch.long)

        all_idx = np.concatenate([tr_idx, te_idx])
        X_sc_all = np.vstack([X_sc_tr, X_sc_te])
        x_tr = torch.tensor(X_sc_tr, dtype=torch.float)
        x_ctx = torch.tensor(X_sc_all, dtype=torch.float)
        y_tr_t = torch.tensor(np.log1p(y_tr), dtype=torch.float)
        n_tr = len(tr_idx)

        tr_t = torch.tensor(tr_idx, dtype=torch.long)
        ei_tr, _ = pyg_subgraph(tr_t, full_ei, relabel_nodes=True, num_nodes=len(df))
        all_t = torch.tensor(all_idx, dtype=torch.long)
        ei_ctx, _ = pyg_subgraph(all_t, full_ei, relabel_nodes=True, num_nodes=len(df))

        mlp = sm.train_nn(sm.MLPModel(X_sc_tr.shape[1]), x_tr, y_tr_t, ei_tr, tp_t, vp_t)
        mlp.eval()
        with torch.no_grad():
            mlp_pred = np.expm1(mlp(x_ctx)[n_tr:].numpy())
        mlp_s = score(y_te, mlp_pred, "MLP")

        log_cap = np.log1p(y_tr.max() * 2)
        gat = sm.train_nn(sm.GATv2Model(X_sc_tr.shape[1]), x_tr, y_tr_t, ei_tr, tp_t, vp_t)
        gat.eval()
        with torch.no_grad():
            raw = gat(x_ctx, ei_ctx)[n_tr:].numpy()
        gat_pred = np.expm1(np.clip(raw, 0, log_cap))
        gat_s = score(y_te, gat_pred, "GATv2")

        log(f"  [{fi+1:2d}/{len(boroughs)}] {b:<30s}  n={len(te_idx):4d}  "
            f"HA={ha['WMAPE']:.3f}  IDW={idw_s['WMAPE']:.3f}  MLR={mlr_s['WMAPE']:.3f}  "
            f"RF={rf_s['WMAPE']:.3f}  XGB={xgb_s['WMAPE']:.3f}  MLP={mlp_s['WMAPE']:.3f}  "
            f"GATv2={gat_s['WMAPE']:.3f}  ({time.time()-t0:.0f}s)")

        base = {"borough": b, "n_test": len(te_idx)}
        rows.extend([base | ha, base | idw_s, base | mlr_s, base | rf_s,
                     base | xgb_s, base | mlp_s, base | gat_s])

    result = pd.DataFrame(rows)
    suffix = "_quick" if QUICK_MODE else ""
    result.to_csv(f"results_cv_alt_target_activity{suffix}.csv", index=False)

    order = ["HistAvg", "IDW", "MLR", "RF", "XGBoost", "MLP", "GATv2"]
    agg = result.groupby("model")[["WMAPE", "RMSE", "MAE"]].agg(["mean", "std", "median"]).round(4)
    agg.columns = [f"{m}_{s}" for m, s in agg.columns]
    agg = agg.reset_index()
    agg["_o"] = agg["model"].map({m: i for i, m in enumerate(order)})
    agg = agg.sort_values("_o").drop(columns="_o")
    agg.to_csv(f"results_summary_alt_target_activity{suffix}.csv", index=False)

    log(f"\n{'='*76}\nRESULTS -- target=total_activity, {len(boroughs)} folds\n{'='*76}")
    for _, r in agg.iterrows():
        log(f"  {r['model']:<10s} WMAPE={r['WMAPE_mean']:.4f}(+-{r['WMAPE_std']:.4f}) med={r['WMAPE_median']:.4f}")
    log("(Reference -- frozen headline, target=total_boardings: HistAvg=1.0822, RF=0.6428, MLP=0.6311, GATv2=0.7187)")
    log(f"\nTotal time: {(time.time()-t_start)/60:.1f} min")
    return agg

run_step4m()


## 4. GATv2 with distance/bearing edge attributes

Gives GATv2's attention mechanism explicit geometric information (Haversine
distance + bearing per edge) that the frozen `GATv2Model` lacks. Trains
plain GATv2 (frozen, unmodified, for a same-run comparison) alongside the
edge-attribute variant. **Needs the raw `data/` folder** (route-edge
construction reads the raw BUSTO CSVs).

In [ ]:
from torch_geometric.nn import GATv2Conv
from torch_geometric.utils import subgraph as pyg_subgraph

EDGE_DIM = 2  # [distance_km, bearing_rad]

def compute_edge_attrs(full_ei, lats, lons):
    src, dst = full_ei[0].numpy(), full_ei[1].numpy()
    lat1, lon1 = np.radians(lats[src]), np.radians(lons[src])
    lat2, lon2 = np.radians(lats[dst]), np.radians(lons[dst])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    dist_km = 2 * 6371.0088 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    bearing = np.arctan2(np.sin(dlon) * np.cos(lat2),
                          np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon))
    raw = np.stack([dist_km, bearing], axis=1).astype(np.float32)
    scaled = StandardScaler().fit_transform(raw)
    return torch.tensor(scaled, dtype=torch.float)


class GATv2EdgeAttrModel(torch.nn.Module):
    """Same architecture as the frozen GATv2Model, extended with edge_dim=2
    so attention is conditioned on distance + bearing, not node features alone."""
    def __init__(self, in_ch, hidden_dim, heads, dropout, edge_dim=EDGE_DIM):
        super().__init__()
        self.c1 = GATv2Conv(in_ch, hidden_dim, heads=heads, dropout=dropout, concat=True, edge_dim=edge_dim)
        self.c2 = GATv2Conv(hidden_dim * heads, 1, heads=1, dropout=dropout, concat=False, edge_dim=edge_dim)
        self.skip = torch.nn.Linear(in_ch, hidden_dim * heads, bias=False)
        self.dropout = dropout

    def forward(self, x, edge_index, edge_attr):
        h = F.elu(self.c1(x, edge_index, edge_attr)) + self.skip(x)
        h = F.dropout(h, p=self.dropout, training=self.training)
        return self.c2(h, edge_index, edge_attr).squeeze(-1)


def train_nn_edge_attrs(model, x, y, ei, ea, tr_pos, val_pos, sm):
    """Mirrors step4_model.train_nn exactly, extended to pass edge_attr."""
    opt = torch.optim.Adam(model.parameters(), lr=sm.LR, weight_decay=1e-4)
    best_val, best_w, no_imp = float("inf"), None, 0
    for epoch in range(sm.EPOCHS):
        model.train(); opt.zero_grad()
        loss = F.huber_loss(model(x, ei, ea)[tr_pos], y[tr_pos], delta=0.5)
        loss.backward(); opt.step()
        if (epoch + 1) % sm.VAL_EVERY != 0:
            continue
        model.eval()
        with torch.no_grad():
            vl = F.huber_loss(model(x, ei, ea)[val_pos], y[val_pos], delta=0.5).item()
        if vl < best_val - 1e-6:
            best_val, best_w, no_imp = vl, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            no_imp += 1
            if no_imp >= sm.PATIENCE:
                break
    if best_w:
        model.load_state_dict(best_w)
    return model


def run_step4n():
    sm = fresh_step4_model("--with-sc")

    df = pd.read_csv(DATA_FILE)
    y_orig = df[TARGET_COL].values.astype(float)
    X_raw = sm.prep_features(df)
    log(f"Features: {X_raw.shape[1]} (AI23+OSM+SC headline set)")

    log(f"Building multigraph ({len(df):,} nodes)...")
    knn_ei = sm.build_knn_edge_index(df["lat"].values, df["lon"].values)
    route_ei = sm.build_route_edges(df)
    full_ei = torch.unique(torch.cat([knn_ei, route_ei], dim=1), dim=1)
    full_ea = compute_edge_attrs(full_ei, df["lat"].values, df["lon"].values)
    log(f"  Combined edges: {full_ei.shape[1]:,}  |  edge_attr shape: {tuple(full_ea.shape)}")

    boroughs = sorted(df[BOROUGH_COL].unique())
    if QUICK_MODE:
        boroughs = boroughs[:5]
    log(f"Running {len(boroughs)}-fold leave-borough-out CV...\n")

    rows = []
    t_start = time.time()
    for fi, b in enumerate(boroughs):
        t0 = time.time()
        test_mask = (df[BOROUGH_COL] == b).values
        tr_idx, te_idx = np.where(~test_mask)[0], np.where(test_mask)[0]
        y_tr, y_te = y_orig[tr_idx], y_orig[te_idx]

        scaler = StandardScaler()
        X_sc_tr = scaler.fit_transform(X_raw[tr_idx])
        X_sc_te = scaler.transform(X_raw[te_idx])

        tp, vp = stratified_val_split(tr_idx, y_tr, sm.SEED, fi, sm.VAL_FRAC)
        tp_t, vp_t = torch.tensor(tp, dtype=torch.long), torch.tensor(vp, dtype=torch.long)

        all_idx = np.concatenate([tr_idx, te_idx])
        X_sc_all = np.vstack([X_sc_tr, X_sc_te])
        x_tr = torch.tensor(X_sc_tr, dtype=torch.float)
        x_ctx = torch.tensor(X_sc_all, dtype=torch.float)
        y_tr_t = torch.tensor(np.log1p(y_tr), dtype=torch.float)
        n_tr = len(tr_idx)

        tr_t = torch.tensor(tr_idx, dtype=torch.long)
        ei_tr, ea_tr = pyg_subgraph(tr_t, full_ei, edge_attr=full_ea, relabel_nodes=True, num_nodes=len(df))
        all_t = torch.tensor(all_idx, dtype=torch.long)
        ei_ctx, ea_ctx = pyg_subgraph(all_t, full_ei, edge_attr=full_ea, relabel_nodes=True, num_nodes=len(df))

        log_cap = np.log1p(y_tr.max() * 2)

        # plain GATv2 (frozen, unmodified) -- same-run comparison point
        gat = sm.train_nn(sm.GATv2Model(X_sc_tr.shape[1]), x_tr, y_tr_t, ei_tr, tp_t, vp_t)
        gat.eval()
        with torch.no_grad():
            raw = gat(x_ctx, ei_ctx)[n_tr:].numpy()
        gat_s = score(y_te, np.expm1(np.clip(raw, 0, log_cap)), "GATv2")

        # GATv2 + edge attributes
        gat_ea_model = GATv2EdgeAttrModel(X_sc_tr.shape[1], sm.HIDDEN_DIM, sm.HEADS, sm.DROPOUT)
        gat_ea = train_nn_edge_attrs(gat_ea_model, x_tr, y_tr_t, ei_tr, ea_tr, tp_t, vp_t, sm)
        gat_ea.eval()
        with torch.no_grad():
            raw_ea = gat_ea(x_ctx, ei_ctx, ea_ctx)[n_tr:].numpy()
        gat_ea_s = score(y_te, np.expm1(np.clip(raw_ea, 0, log_cap)), "GATv2-EdgeAttr")

        log(f"  [{fi+1:2d}/{len(boroughs)}] {b:<30s}  n={len(te_idx):4d}  "
            f"GATv2={gat_s['WMAPE']:.3f}  GATv2-EdgeAttr={gat_ea_s['WMAPE']:.3f}  ({time.time()-t0:.0f}s)")

        base = {"borough": b, "n_test": len(te_idx)}
        rows.extend([base | gat_s, base | gat_ea_s])

    result = pd.DataFrame(rows)
    suffix = "_quick" if QUICK_MODE else ""
    result.to_csv(f"results_cv_gatv2_edge_attrs{suffix}.csv", index=False)

    agg = result.groupby("model")[["WMAPE", "RMSE", "MAE"]].agg(["mean", "std", "median"]).round(4)
    agg.columns = [f"{m}_{s}" for m, s in agg.columns]
    agg = agg.reset_index()
    agg.to_csv(f"results_summary_gatv2_edge_attrs{suffix}.csv", index=False)

    log(f"\n{'='*76}\nRESULTS -- GATv2 vs GATv2+EdgeAttr, {len(boroughs)} folds\n{'='*76}")
    for _, r in agg.iterrows():
        log(f"  {r['model']:<18s} WMAPE={r['WMAPE_mean']:.4f}(+-{r['WMAPE_std']:.4f}) med={r['WMAPE_median']:.4f}")
    log("(Reference -- frozen headline: GATv2=0.7187, GCN=0.7006, MLP=0.6311)")
    log(f"\nTotal time: {(time.time()-t_start)/60:.1f} min")
    return agg

run_step4n()


## 5. Stacked tabular ensemble (MLR + RF + XGBoost + MLP)

Blends the four near-tied tabular models (excludes GATv2/graph models -- 7
independent graph interventions already lost to tabular baselines). Uses a
leakage-safe design: base models fit on 90% of each fold's training data
(`tp`), blend weights grid-searched on the held-out 10% (`vp`), then both
the fitted-weight blend and a plain equal-weight blend are scored on the
true test borough. 3 seeds (42, 142, 242).

In [ ]:
def grid_search_weights(preds_val, y_val_log, step=0.1):
    names = list(preds_val.keys())
    ticks = int(round(1 / step))
    best_w, best_err = None, float("inf")
    for combo in itertools.product(range(ticks + 1), repeat=len(names)):
        if sum(combo) != ticks:
            continue
        w = np.array(combo, dtype=float) / ticks
        blend = sum(w[i] * preds_val[names[i]] for i in range(len(names)))
        err = float(np.mean((blend - y_val_log) ** 2))
        if err < best_err:
            best_err, best_w = err, w
    return dict(zip(names, best_w))


def train_nn_logged(model, x, y, tr_pos, val_pos, sm):
    """Copy of step4_model.train_nn, instrumented to also return the stop epoch."""
    opt = torch.optim.Adam(model.parameters(), lr=sm.LR, weight_decay=1e-4)
    best_val, best_w, no_imp, stop_epoch = float("inf"), None, 0, sm.EPOCHS
    for epoch in range(sm.EPOCHS):
        model.train(); opt.zero_grad()
        loss = F.huber_loss(model(x, None)[tr_pos], y[tr_pos], delta=0.5)
        loss.backward(); opt.step()
        if (epoch + 1) % sm.VAL_EVERY != 0:
            continue
        model.eval()
        with torch.no_grad():
            vl = F.huber_loss(model(x, None)[val_pos], y[val_pos], delta=0.5).item()
        if vl < best_val - 1e-6:
            best_val, best_w, no_imp = vl, {k: v.clone() for k, v in model.state_dict().items()}, 0
        else:
            no_imp += 1
            if no_imp >= sm.PATIENCE:
                stop_epoch = epoch + 1
                break
    if best_w:
        model.load_state_dict(best_w)
    return model, stop_epoch


def run_step4o():
    sm = fresh_step4_model("--with-sc")
    SEEDS = [42, 142, 242]
    EPOCH_LOG_SEED = 42

    df = pd.read_csv(DATA_FILE)
    y_orig = df[TARGET_COL].values.astype(float)
    X_raw = sm.prep_features(df)
    log(f"Features: {X_raw.shape[1]} (AI23+OSM+SC headline set)")

    boroughs = sorted(df[BOROUGH_COL].unique())
    if QUICK_MODE:
        boroughs = boroughs[:5]
    log(f"Running {len(boroughs)} folds x {len(SEEDS)} seeds (pure-tabular ensemble)\n")

    all_rows, weight_rows = [], []
    t_start = time.time()
    for seed in SEEDS:
        log(f"--- seed={seed} ---")
        torch.manual_seed(seed); np.random.seed(seed)
        log_epochs = (seed == EPOCH_LOG_SEED)
        for fi, b in enumerate(boroughs):
            t0 = time.time()
            test_mask = (df[BOROUGH_COL] == b).values
            tr_idx, te_idx = np.where(~test_mask)[0], np.where(test_mask)[0]
            y_tr, y_te = y_orig[tr_idx], y_orig[te_idx]

            tp, vp = stratified_val_split(tr_idx, y_tr, seed, fi, sm.VAL_FRAC)
            X_tp_raw, X_vp_raw, X_te_raw = X_raw[tr_idx][tp], X_raw[tr_idx][vp], X_raw[te_idx]
            y_tp, y_vp = y_tr[tp], y_tr[vp]
            y_tp_log, y_vp_log = np.log1p(y_tp), np.log1p(y_vp)

            scaler = sm.StandardScaler()
            X_tp_sc = scaler.fit_transform(X_tp_raw)   # fit on tp ONLY
            X_vp_sc = scaler.transform(X_vp_raw)
            X_te_sc = scaler.transform(X_te_raw)

            mlr = Ridge(alpha=1.0, random_state=seed)
            mlr.fit(X_tp_sc, y_tp_log)
            rf = RandomForestRegressor(n_estimators=sm.RF_TREES, max_features="sqrt",
                                        min_samples_leaf=5, n_jobs=-1, random_state=seed)
            rf.fit(X_tp_sc, y_tp_log)
            xgbr = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05,
                                 random_state=seed, n_jobs=-1, verbosity=0)
            xgbr.fit(X_tp_sc, y_tp_log)

            x_tp_t = torch.tensor(X_tp_sc, dtype=torch.float)
            x_vp_t = torch.tensor(X_vp_sc, dtype=torch.float)
            x_te_t = torch.tensor(X_te_sc, dtype=torch.float)
            y_tp_t = torch.tensor(y_tp_log, dtype=torch.float)
            y_vp_t = torch.tensor(y_vp_log, dtype=torch.float)
            x_fit = torch.cat([x_tp_t, x_vp_t], dim=0)
            y_fit = torch.cat([y_tp_t, y_vp_t], dim=0)
            tp_pos = torch.arange(len(tp), dtype=torch.long)
            vp_pos = torch.arange(len(tp), len(tp) + len(vp), dtype=torch.long)

            mlp, stop_epoch = train_nn_logged(sm.MLPModel(X_tp_sc.shape[1]), x_fit, y_fit, tp_pos, vp_pos, sm)
            mlp.eval()
            with torch.no_grad():
                mlp_vp_log = mlp(x_vp_t, None).numpy()
                mlp_te_log = mlp(x_te_t, None).numpy()

            preds_vp = {"MLR": mlr.predict(X_vp_sc), "RF": rf.predict(X_vp_sc),
                        "XGBoost": xgbr.predict(X_vp_sc), "MLP": mlp_vp_log}
            weights = grid_search_weights(preds_vp, y_vp_log)

            preds_te = {"MLR": mlr.predict(X_te_sc), "RF": rf.predict(X_te_sc),
                        "XGBoost": xgbr.predict(X_te_sc), "MLP": mlp_te_log}
            fitted_blend_log = sum(weights[k] * preds_te[k] for k in preds_te)
            equal_blend_log = sum(preds_te[k] for k in preds_te) / len(preds_te)

            ha_s = score(y_te, np.full(len(te_idx), y_tr.mean()), "HistAvg")
            mlr_s = score(y_te, np.expm1(preds_te["MLR"]), "MLR")
            rf_s = score(y_te, np.expm1(preds_te["RF"]), "RF")
            xgb_s = score(y_te, np.expm1(preds_te["XGBoost"]), "XGBoost")
            mlp_s = score(y_te, np.expm1(preds_te["MLP"]), "MLP")
            ensf_s = score(y_te, np.expm1(fitted_blend_log), "EnsembleFitted")
            ense_s = score(y_te, np.expm1(equal_blend_log), "EnsembleEqual")

            epoch_note = f"  epochs={stop_epoch}" if log_epochs else ""
            log(f"  [{fi+1:2d}/{len(boroughs)}] {b:<30s}  n={len(te_idx):4d}  "
                f"MLR={mlr_s['WMAPE']:.3f}  RF={rf_s['WMAPE']:.3f}  XGB={xgb_s['WMAPE']:.3f}  "
                f"MLP={mlp_s['WMAPE']:.3f}  ENSfit={ensf_s['WMAPE']:.3f}  ENSeq={ense_s['WMAPE']:.3f}"
                f"  ({time.time()-t0:.0f}s){epoch_note}")

            base = {"seed": seed, "borough": b, "n_test": len(te_idx)}
            all_rows.extend([base | ha_s, base | mlr_s, base | rf_s, base | xgb_s,
                              base | mlp_s, base | ensf_s, base | ense_s])
            weight_rows.append({"seed": seed, "borough": b, "w_MLR": weights["MLR"],
                                 "w_RF": weights["RF"], "w_XGBoost": weights["XGBoost"],
                                 "w_MLP": weights["MLP"], "mlp_stop_epoch": stop_epoch})

    results = pd.DataFrame(all_rows)
    suffix = "_quick" if QUICK_MODE else ""
    results.to_csv(f"results_cv_tabular_ensemble{suffix}.csv", index=False)
    pd.DataFrame(weight_rows).to_csv(f"results_ensemble_weights{suffix}.csv", index=False)

    order = ["HistAvg", "MLR", "RF", "XGBoost", "MLP", "EnsembleFitted", "EnsembleEqual"]
    agg_mean = results.groupby("model")[["WMAPE", "RMSE", "MAE"]].agg(["mean", "std", "median"]).round(4)
    agg_mean.columns = [f"{m}_{s}" for m, s in agg_mean.columns]
    agg = agg_mean.reset_index()
    agg["_o"] = agg["model"].map({m: i for i, m in enumerate(order)})
    agg = agg.sort_values("_o").drop(columns="_o")
    agg.to_csv(f"results_summary_tabular_ensemble{suffix}.csv", index=False)

    log(f"\n{'='*76}\nRESULTS -- {len(boroughs)} folds x {len(SEEDS)} seeds\n{'='*76}")
    for _, r in agg.iterrows():
        log(f"  {r['model']:<10s}  WMAPE={r['WMAPE_mean']:.4f}(+/-{r['WMAPE_std']:.4f})  med={r['WMAPE_median']:.4f}")
    log("(Reference -- frozen single-seed headline: HistAvg=1.0822, MLR=0.6404, RF=0.6428, XGBoost=0.6437, MLP=0.6311)")

    log(f"\n{'='*76}\nSIGNIFICANCE CHECK vs MLP (Wilcoxon, seed-averaged per borough, n={len(boroughs)})\n{'='*76}")
    borough_seed_avg = results.groupby(["model", "borough"])["WMAPE"].mean().unstack(level=0)
    mlp_vec = borough_seed_avg["MLP"].values
    tests = []
    for name in ["EnsembleFitted", "EnsembleEqual"]:
        vec = borough_seed_avg[name].values
        stat, p = wilcoxon(vec, mlp_vec)
        tests.append((name, p, np.mean(vec - mlp_vec)))
    for rank, (name, p, diff) in enumerate(sorted(tests, key=lambda t: t[1]), start=1):
        thresh = 0.05 / (2 - rank + 1)
        sig = p < thresh
        log(f"  {name} vs MLP: mean diff={diff:+.4f}  p={p:.5f}  Holm-Bonferroni threshold={thresh:.4f}  "
            f"{'SIGNIFICANT' if sig else 'not significant'}")
        if not sig:
            log("  (Holm-Bonferroni stops at first non-rejection)")
            break

    if not QUICK_MODE:
        ens_epochs = pd.DataFrame(weight_rows)
        ens_epochs = ens_epochs[ens_epochs["seed"] == EPOCH_LOG_SEED]["mlp_stop_epoch"]
        log(f"\nMLP epoch-count diagnostic (seed={EPOCH_LOG_SEED}, {len(ens_epochs)} folds):")
        log(f"  mean={ens_epochs.mean():.0f}  median={ens_epochs.median():.0f}  "
            f"min={ens_epochs.min()}  max={ens_epochs.max()}  (ceiling={sm.EPOCHS})")

    log(f"\nTotal time: {(time.time()-t_start)/60:.1f} min")
    return agg

run_step4o()
